## Ingest drivers.json


### Step 0 - Parameters and Imports


In [0]:
%run "../utils/config"


In [0]:
dbutils.widgets.text("p_source_value", "")
v_source_value = dbutils.widgets.get("p_source_value")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType, TimestampType, FloatType
from pyspark.sql.functions import col, lit, current_timestamp, to_timestamp, concat

### Step 1 - Read the json file


#### 1.1 Define the schema


In [0]:
name_schema = StructType([
    StructField("forename", StringType(), False),
    StructField("surname", StringType(), False)
])


In [0]:
drivers_schema = StructType([
    StructField("code", StringType(), False),
    StructField("dob", DateType(), True),
    StructField("driverId", IntegerType(), True),
    StructField("driverRef", StringType(), True),
    StructField("name", name_schema, True),
    StructField("nationality", StringType(), True),
    StructField("number", IntegerType(), True),
    StructField("url", StringType(), True)
])


#### 1.2 Read the json file


In [ ]:
drivers_df = (
    spark.read
        .format("json")
        .schema(drivers_schema)
        .load(f"{raw_folder_path}/drivers.json")
        )


### Step 2 - Transform the data


In [0]:

drivers_transformed_df = add_ingestion_date(
    drivers_df
        .withColumnRenamed("driverId", "driver_id")
        .withColumnRenamed("driverRef", "driver_ref")
        .withColumn("name",concat(col("name.forename"),lit(" "), col("name.surname")))
        )


In [0]:
dirvers_renamed_df = drivers_transformed_df.drop(col("url")).withColumn("source", lit(v_source_value))


### Step 3 - Write the output to parquet


In [ ]:
dirvers_renamed_df.write.format("delta").mode("overwrite").saveAsTable("f1_processed.drivers")


In [0]:
dbutils.notebook.exit("success")